In [4]:
! pip install playwright beautifulsoup4 pandas requests
! playwright install

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.9/41.9 MB 5.0 MB/s  0:00:08m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [playwright]3 [playwright]
165.5 MiB [                    ] 0% 0.0s165.5 MiB [                    ] 0% 189.0s165.5 MiB [                    ] 0% 2094.1s165.5 MiB [                    ] 0% 1317.4s165.5 MiB [                    ] 0% 1506.4s165.5 MiB [                    ] 0% 1650.2s165.5 MiB [                    ] 0% 1383.0s165.5 MiB [                    ] 0% 1383.1s165.5 MiB [                    ] 0% 1209.0s165.5 MiB [                    ] 0% 1561.7s165.5 MiB [                    ] 0% 1377.0s165.5 MiB [                    ] 0% 1227.3s165.5 MiB [                    ] 0% 1249.6s165.5 MiB [                    ] 0% 1147.5s165.5 MiB [                    ] 0% 1085.7s165.5 MiB [                    ] 0% 1033.1s165.5 MiB [                    ] 0% 974.6s165.5 MiB [                    ] 0% 927.7s165.5 MiB [                    ] 0% 885.5s165.5 MiB [                

In [6]:
from playwright.sync_api import sync_playwright
from bs4 import BeautifulSoup
from urllib.parse import urljoin
import requests
import os
import re
import pandas as pd

BASE_URL = "https://www.cse.lk"
REPORTS_URL = "https://www.cse.lk/?page=financial-reports"

PDF_DIR = "data/raw_pdfs"
META_PATH = "data/metadata/cse_financial_reports.csv"

os.makedirs(PDF_DIR, exist_ok=True)
os.makedirs(os.path.dirname(META_PATH), exist_ok=True)

def safe_filename(text):
    text = re.sub(r"[^\w\s.-]", "", text)
    return re.sub(r"\s+", "_", text.strip())[:180]

def download_pdf(url, path):
    r = requests.get(url, timeout=30, headers={"User-Agent": "Mozilla/5.0"})
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)

def collect_cse_financial_reports():
    records = []

    with sync_playwright() as p:
        browser = p.chromium.launch(headless=True)
        page = browser.new_page()
        page.goto(REPORTS_URL, wait_until="networkidle")

        html = page.content()
        browser.close()

    soup = BeautifulSoup(html, "html.parser")

    for a in soup.find_all("a", href=True):
        href = a["href"]

        if ".pdf" not in href.lower():
            continue

        pdf_url = urljoin(BASE_URL, href)
        title = a.get_text(strip=True) or pdf_url.split("/")[-1]
        filename = safe_filename(title) + ".pdf"
        local_path = os.path.join(PDF_DIR, filename)

        download_pdf(pdf_url, local_path)

        records.append({
            "source": "CSE",
            "category": "financial_report",
            "title": title,
            "pdf_url": pdf_url,
            "local_path": local_path
        })

    df = pd.DataFrame(records).drop_duplicates("pdf_url")
    df.to_csv(META_PATH, index=False)
    return df